# CO₂ Removal from Natural Gas using Amine Absorption (MDEA)

This notebook demonstrates CO₂ removal from natural gas using **MDEA (methyldiethanolamine)**
amine absorption in NeqSim.

## Process Overview

Amine absorption is the most common method for removing CO₂ from natural gas. The process uses
a chemical solvent (MDEA) that reacts reversibly with CO₂:

$$\text{MDEA} + \text{CO}_2 + \text{H}_2\text{O} \rightleftharpoons \text{MDEA}^+ + \text{HCO}_3^-$$

### Typical Process
1. **Absorber Column** — Sour gas contacts lean MDEA solution (40-50°C, high pressure)
2. **Regenerator (Stripper)** — Rich MDEA is heated to release CO₂ (100-120°C, low pressure)
3. **Lean/Rich Heat Exchanger** — Heat recovery between lean and rich amine streams

## Topics Covered
- CO₂ solubility in MDEA solutions (vapor-liquid equilibrium)
- CO₂ loading curves at different temperatures
- Simple absorption process simulation
- Effect of MDEA concentration and circulation rate

In [ ]:
# Setup NeqSim
import subprocess, sys
try:
    from neqsim import jneqsim
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'neqsim'])
    from neqsim import jneqsim

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

SystemFurstElectrolyteEos = jneqsim.thermo.system.SystemFurstElectrolyteEos
SystemSrkEos = jneqsim.thermo.system.SystemSrkEos
ThermodynamicOperations = jneqsim.thermodynamicoperations.ThermodynamicOperations
Stream = jneqsim.process.equipment.stream.Stream
ProcessSystem = jneqsim.process.processmodel.ProcessSystem

print('NeqSim loaded successfully')

## 1. CO₂ Solubility in MDEA Solutions

Calculate CO₂ vapor-liquid equilibrium with aqueous MDEA at different temperatures.
The **electrolyte EOS (Furst)** handles the chemical reactions between CO₂ and MDEA.

In [ ]:
# CO2 loading curve - CO2 partial pressure vs loading at different temperatures
temperatures = [40, 60, 80, 100]  # °C
mdea_wt_pct = 50.0  # wt% MDEA in solvent
total_P = 50.0  # bara

fig, ax = plt.subplots(figsize=(10, 7))
colors = ['blue', 'green', 'orange', 'red']

for temp, color in zip(temperatures, colors):
    co2_partial_pressures = []
    co2_loadings = np.arange(0.01, 0.8, 0.05)
    valid_loadings = []

    for loading in co2_loadings:
        try:
            fluid = SystemFurstElectrolyteEos(273.15 + float(temp), total_P)
            # Add gas phase component
            fluid.addComponent('methane', 10.0)  # Carrier gas
            # Add MDEA solution
            fluid.addComponent('MDEA', 1.0)  # 1 mol MDEA
            fluid.addComponent('water', 5.0)  # ~50 wt% MDEA solution
            # Add CO2 based on loading (mol CO2 / mol MDEA)
            fluid.addComponent('CO2', float(loading) * 1.0)
            fluid.chemicalReactionInit()
            fluid.setMixingRule(4)

            ops = ThermodynamicOperations(fluid)
            ops.TPflash()
            fluid.initProperties()

            # Get CO2 partial pressure in gas phase
            if fluid.getNumberOfPhases() > 1:
                x_co2_gas = float(fluid.getPhase('gas').getComponent('CO2').getx())
                pp_co2 = x_co2_gas * total_P
                co2_partial_pressures.append(pp_co2)
                valid_loadings.append(float(loading))
        except Exception:
            pass

    if valid_loadings:
        ax.semilogy(valid_loadings, co2_partial_pressures, 'o-',
                    color=color, linewidth=2, markersize=5, label=f'{temp}°C')

ax.set_xlabel('CO₂ Loading (mol CO₂ / mol MDEA)', fontsize=12)
ax.set_ylabel('CO₂ Partial Pressure (bara)', fontsize=12)
ax.set_title(f'CO₂ Vapor-Liquid Equilibrium in {mdea_wt_pct:.0f} wt% MDEA Solution')
ax.legend(fontsize=11, title='Temperature')
ax.grid(True, which='both', alpha=0.3)
ax.set_xlim(0, 0.8)
plt.tight_layout()
plt.savefig('co2_mdea_vle.png', dpi=150, bbox_inches='tight')
plt.show()

## 2. Effect of Temperature on CO₂ Absorption

Compare the CO₂ content remaining in gas phase at different absorber temperatures,
demonstrating why lower temperatures favor absorption.

In [ ]:
# Temperature sweep for fixed feed composition
temps = np.arange(25, 95, 5)
co2_in_gas = []
co2_in_liquid = []

for t in temps:
    try:
        fluid = SystemFurstElectrolyteEos(273.15 + float(t), 50.0)
        fluid.addComponent('methane', 80.0)
        fluid.addComponent('CO2', 10.0)  # 10% sour gas
        fluid.addComponent('MDEA', 5.0)
        fluid.addComponent('water', 25.0)
        fluid.chemicalReactionInit()
        fluid.setMixingRule(4)

        ops = ThermodynamicOperations(fluid)
        ops.TPflash()
        fluid.initProperties()

        if fluid.getNumberOfPhases() > 1:
            x_co2_g = float(fluid.getPhase('gas').getComponent('CO2').getx()) * 100
            co2_in_gas.append(x_co2_g)
        else:
            co2_in_gas.append(float('nan'))
    except Exception:
        co2_in_gas.append(float('nan'))

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(temps, co2_in_gas, 'r-o', linewidth=2, markersize=5)
ax.set_xlabel('Temperature (°C)', fontsize=12)
ax.set_ylabel('CO₂ in Gas Phase (mol%)', fontsize=12)
ax.set_title('CO₂ in Treated Gas vs Absorber Temperature (50 bara, 50 wt% MDEA)')
ax.grid(alpha=0.3)
ax.axhline(y=2.0, color='green', linestyle='--', alpha=0.7, label='2% CO₂ spec')
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig('co2_temp_effect.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Process Simulation: CO₂ Absorption with MDEA

Simulate the full absorption process using the `SimpleAbsorber` equipment class.

In [ ]:
import jpype

SimpleAbsorber = jpype.JClass('neqsim.process.equipment.absorber.SimpleAbsorber')

# Sour gas feed (5% CO2 in natural gas)
sour_gas_fluid = SystemFurstElectrolyteEos(273.15 + 40.0, 60.0)
sour_gas_fluid.addComponent('methane', 85.0)
sour_gas_fluid.addComponent('ethane', 5.0)
sour_gas_fluid.addComponent('propane', 3.0)
sour_gas_fluid.addComponent('CO2', 5.0)
sour_gas_fluid.addComponent('nitrogen', 2.0)
sour_gas_fluid.chemicalReactionInit()
sour_gas_fluid.setMixingRule(4)

sour_gas = Stream('Sour Gas Feed', sour_gas_fluid)
sour_gas.setFlowRate(100000.0, 'kg/hr')  # 100 t/h
sour_gas.setTemperature(40.0, 'C')
sour_gas.setPressure(60.0, 'bara')

# Lean MDEA solvent
lean_mdea_fluid = SystemFurstElectrolyteEos(273.15 + 40.0, 60.0)
lean_mdea_fluid.addComponent('MDEA', 50.0)  # ~50 wt% MDEA
lean_mdea_fluid.addComponent('water', 50.0)
lean_mdea_fluid.addComponent('CO2', 0.5)  # residual CO2
lean_mdea_fluid.chemicalReactionInit()
lean_mdea_fluid.setMixingRule(4)

lean_mdea = Stream('Lean MDEA', lean_mdea_fluid)
lean_mdea.setFlowRate(200000.0, 'kg/hr')  # High solvent rate
lean_mdea.setTemperature(40.0, 'C')
lean_mdea.setPressure(60.0, 'bara')

process = ProcessSystem()
process.add(sour_gas)
process.add(lean_mdea)

# Absorber
absorber = SimpleAbsorber('CO2 Absorber', sour_gas)
absorber.setAproachToEquilibrium(0.75)
process.add(absorber)

process.run()

print('=== CO₂ Absorption with MDEA ===')
print(f'Sour gas feed rate:   {sour_gas.getFlowRate("kg/hr"):.0f} kg/hr')

# Get outlet compositions
gas_out = absorber.getGasOutStream()
liq_out = absorber.getLiquidOutStream()

gas_fluid = gas_out.getFluid()
print(f'\nTreated Gas Composition:')
for i in range(int(gas_fluid.getNumberOfComponents())):
    comp = gas_fluid.getPhase(0).getComponent(i)
    x = float(comp.getx()) * 100
    if x > 0.01:
        print(f'  {str(comp.getComponentName()):12s}: {x:7.3f} mol%')

print(f'\nTreated gas rate: {gas_out.getFlowRate("kg/hr"):.0f} kg/hr')
print(f'Rich MDEA rate:   {liq_out.getFlowRate("kg/hr"):.0f} kg/hr')

## 4. Effect of MDEA Circulation Rate

Vary the lean MDEA flow rate to observe the impact on CO₂ removal efficiency.

In [ ]:
# Circulation rate study
mdea_rates = np.arange(50000, 400000, 25000)  # kg/hr
co2_treated = []
co2_removal_pct = []

for rate in mdea_rates:
    try:
        sg_f = SystemFurstElectrolyteEos(273.15 + 40.0, 60.0)
        sg_f.addComponent('methane', 85.0)
        sg_f.addComponent('ethane', 5.0)
        sg_f.addComponent('CO2', 5.0)
        sg_f.addComponent('nitrogen', 5.0)
        sg_f.chemicalReactionInit()
        sg_f.setMixingRule(4)

        sg = Stream('SG', sg_f)
        sg.setFlowRate(100000.0, 'kg/hr')
        sg.setTemperature(40.0, 'C')
        sg.setPressure(60.0, 'bara')

        lm_f = SystemFurstElectrolyteEos(273.15 + 40.0, 60.0)
        lm_f.addComponent('MDEA', 50.0)
        lm_f.addComponent('water', 50.0)
        lm_f.addComponent('CO2', 0.5)
        lm_f.chemicalReactionInit()
        lm_f.setMixingRule(4)

        lm = Stream('LM', lm_f)
        lm.setFlowRate(float(rate), 'kg/hr')
        lm.setTemperature(40.0, 'C')
        lm.setPressure(60.0, 'bara')

        proc = ProcessSystem()
        proc.add(sg)
        proc.add(lm)

        abs_col = SimpleAbsorber('Absorber', sg)
        abs_col.setAproachToEquilibrium(0.75)
        proc.add(abs_col)
        proc.run()

        gas_out_f = abs_col.getGasOutStream().getFluid()
        co2_out = float(gas_out_f.getPhase(0).getComponent('CO2').getx()) * 100
        co2_treated.append(co2_out)
    except Exception as e:
        co2_treated.append(float('nan'))

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(np.array(mdea_rates)/1000, co2_treated, 'b-o', linewidth=2, markersize=6)
ax.set_xlabel('MDEA Circulation Rate (t/h)', fontsize=12)
ax.set_ylabel('CO₂ in Treated Gas (mol%)', fontsize=12)
ax.set_title('CO₂ Removal vs MDEA Circulation Rate')
ax.axhline(y=2.0, color='green', linestyle='--', alpha=0.7, label='Typical CO₂ spec (2 mol%)')
ax.legend(fontsize=11)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('co2_mdea_circulation.png', dpi=150, bbox_inches='tight')
plt.show()

## Summary

Key observations:

- **MDEA** selectively absorbs CO₂ through chemical reaction (protonation + bicarbonate formation)
- **Temperature**: Lower absorber temperature favors CO₂ absorption (30-50°C typical)
- **Pressure**: Higher pressure increases CO₂ partial pressure and driving force
- **MDEA concentration**: 40-50 wt% MDEA is standard in industry
- **Circulation rate**: Higher solvent rate improves removal but increases energy cost

### NeqSim Features Used
- `SystemFurstElectrolyteEos` — Handles chemical equilibria (MDEA protonation, HCO₃⁻)
- `chemicalReactionInit()` — Initializes reaction database for the system
- `SimpleAbsorber` — Single-stage absorption with approach to equilibrium

### Industrial Considerations
- Regeneration energy is the largest OPEX item (~3-4 GJ/t CO₂)
- MDEA degradation requires reclaiming and makeup
- Foaming in absorber reduces efficiency
- Corrosion monitoring critical in hot sections